# Seed-labeling campaign QC and near-duplicate validation

Two things are measured here, and they share a theme: **a design that can be audited before
it has produced any data.**

The first is the **seed-labeling campaign** (stage s07). 12,000 prompts are drawn from the
universe, stratified by source × size band, with a deficit-redistribution rule for sources
that cannot fill their quota. 100 of them become a calibration batch that a human revises
by hand; the remaining 11,900 are sliced into batches of 77 new items plus **3 gold items
carrying no mark of any kind** in the file handed to the annotator — which of the 80 are
gold lives only in the manifest, because marking them would measure an agent's care on
three items rather than the quality of the batch. Agreement is scored on `task_type` and
`domain` of those 3 golds, so six comparisons at a granularity of 1/6, and the gate is
0.80 — five of six. The `manifest.json` is the campaign's **versioned ledger**: it is the
one file in the labeling tree that is committed, precisely so that the design is
inspectable before, during and after the run.

The second is the **near-duplicate validation** of stage s06, which *has* run, and whose
output on disk supports an exact invariant check.

**Read-only, and no prompt text.** Both `labeling/seed/seed.parquet` and
`labeling/batches/*.json` carry full prompt bodies and are gitignored for that reason. This
notebook reads the seed's *metadata columns only* — it never loads the `text` column — and
renders counts and distributions.

In [ ]:
# O backend TEM de ser o `inline`: ele é headless (renderiza por baixo com o
# mesmo Agg) E devolve a figura como saída da célula, que é o que o nbconvert
# embute no HTML. `matplotlib.use("Agg")` puro faz `plt.show()` virar no-op — o
# notebook executa limpo, sai com código 0 e produz um HTML sem UM gráfico.
%matplotlib inline
import base64
import io
import json
from collections import Counter
from html import escape as escape_html

import matplotlib
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from IPython.display import HTML, display

# png e não svg: o histograma do near-dup tem dezenas de milhares de patches, e
# em svg o HTML commitado sairia com megabytes de vetor.
%config InlineBackend.figure_formats = ["png"]

from prompt_factory import paths, schema

AZUL, LARANJA, CINZA, VERDE, VERMELHO = "#2f6f9f", "#d98032", "#8c8c8c", "#4a8c5f", "#b04a4a"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
    "figure.facecolor": "white", "axes.facecolor": "white",
})


def mostrar(fig, alt):
    # A figura é embutida À MÃO, com o `alt` escrito por nós. `plt.show()` sob
    # `matplotlib.use("Agg")` é no-op (HTML sem gráfico nenhum, sem erro), e o
    # `display(fig, metadata={"image/png": {"alt": ...}})` desenha a figura e
    # descarta o alt — o bloco `data_png` do template `lab` do nbconvert não lê
    # essa chave. Emitir o <img> daqui mantém o alt real e o render em UM comando.
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    plt.close(fig)          # senão o backend inline redesenha a figura no fim da célula
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    display(HTML(f'<img alt="{escape_html(alt, quote=True)}" '
                 f'src="data:image/png;base64,{b64}" '
                 f'style="max-width:100%;height:auto">'))


def anotar(ax, valores, ys, *, dx, fmt="{:,.0f}"):
    for v, y in zip(valores, ys, strict=True):
        ax.annotate(fmt.format(v), (v + dx, y), va="center", ha="left",
                    fontsize=8.5, color="#333")


# Flags de existência: nada aqui pode levantar. Um clone limpo tem o manifest e a
# taxonomia (versionados) e NÃO tem a semente nem os parquets (regeneráveis).
SEED_PARQUET = paths.SEED / "seed.parquet"
STRATA_TXT = paths.SEED / "strata.txt"
SEED_LABELS = paths.FINAL / "seed_labels.parquet"
NEAR_MAP = paths.FINAL / "dedup_near_map.parquet"
EXACT_MAP = paths.INTERIM / "dedup_exact_map.parquet"
UNIVERSE = paths.FINAL / "universe.parquet"
DEDUP1 = paths.INTERIM / "dedup1.parquet"
NORMALIZED = paths.INTERIM / "normalized.parquet"
LANG_PARQUET = paths.INTERIM / "lang.parquet"

print("matplotlib backend:", matplotlib.get_backend())
assert "inline" in matplotlib.get_backend().lower(), (
    "backend sem captura de figura: o HTML sairia sem gráfico nenhum")

TEM_MANIFEST = paths.LABEL_MANIFEST.exists()
manifesto = json.loads(paths.LABEL_MANIFEST.read_text(encoding="utf-8")) if TEM_MANIFEST else {}
taxonomia = schema.load_taxonomy()

for rotulo, caminho in (
    ("labeling/manifest.json", paths.LABEL_MANIFEST),
    ("labeling/taxonomy.json", paths.TAXONOMY_JSON),
    ("labeling/seed/seed.parquet", SEED_PARQUET),
    ("labeling/seed/strata.txt", STRATA_TXT),
    ("data/final/seed_labels.parquet", SEED_LABELS),
    ("data/final/dedup_near_map.parquet", NEAR_MAP),
    ("data/interim/dedup_exact_map.parquet", EXACT_MAP),
    ("data/final/universe.parquet", UNIVERSE),
):
    marca = "ok     " if caminho.exists() else "ABSENT "
    tam = f"{caminho.stat().st_size:>13,} bytes" if caminho.exists() else ""
    print(f"  [{marca}] {rotulo:<38} {tam}")

print(f"\ntaxonomy v{taxonomia['version']}: "
      f"{len(taxonomia['task_type'])} task types, {len(taxonomia['domain'])} domains, "
      f"{len(taxonomia['flags'])} flags")

## 1. The campaign design, as actually laid out

Quotas are planned per source, then reconciled against what the universe can actually
supply. A source that cannot fill its quota **cedes the deficit** to the others in
proportion, and every cession is written into `strata.txt` and into the manifest's
`allocation.notas` — so the design is not a plan that diverged silently from the artifact,
it is a plan whose divergences are part of the artifact.

The `oasst` row is the case that makes this a normal path rather than an exception: a quota
of 600 against far fewer available rows in the universe.

In [ ]:
if TEM_MANIFEST:
    aloc = manifesto.get("allocation", {})
    print(f"seed size: {manifesto.get('n_seed'):,}   rng_seed: {manifesto.get('seed_rng')}   "
          f"size bands (chars): {manifesto.get('bucket_edges')}   "
          f"truncation for the agent: {manifesto.get('truncate_chars')} chars")

    fig, axes = plt.subplots(1, len(aloc), figsize=(11.5, 4.2))
    axes = axes if len(aloc) > 1 else [axes]
    for ax, (lang, bloco) in zip(axes, aloc.items(), strict=True):
        fontes = list(bloco["final"])
        y = range(len(fontes))
        planejado = [bloco["planejado"].get(f, 0) for f in fontes]
        final = [bloco["final"].get(f, 0) for f in fontes]
        disponivel = [bloco["disponivel"].get(f, 0) for f in fontes]
        alt = 0.38
        b1 = ax.barh([i + alt / 2 for i in y], planejado, alt, color=CINZA, label="planned quota")
        b2 = ax.barh([i - alt / 2 for i in y], final, alt, color=AZUL, label="final allocation")
        for i, (p, f_, d) in enumerate(
                zip(planejado, final, disponivel, strict=True)):
            ax.annotate(f"{p:,}", (p + 40, i + alt / 2), va="center", fontsize=8, color="#555")
            ax.annotate(f"{f_:,}", (f_ + 40, i - alt / 2), va="center", fontsize=8,
                        color=VERMELHO if f_ < p else "#333",
                        weight="bold" if f_ < p else "normal")
            if d < p:                       # a fonte que não conseguiu encher a cota
                ax.annotate(f"only {d:,} available", (max(p, f_) + 260, i),
                            va="center", fontsize=8, color=VERMELHO, style="italic")
        ax.set_yticks(list(y), fontes)
        ax.invert_yaxis()
        ax.set_xlim(0, max(planejado + final) * 1.55)
        ax.set_xlabel("prompts in the seed")
        ax.set_title(f"{lang} — target {sum(final):,} (n = {sum(final):,})")
        ax.legend(loc="lower right", frameon=False, fontsize=8.5)
    plt.tight_layout()
    mostrar(fig, "Two panels, Portuguese and English, each a horizontal bar chart comparing the planned quota with the final allocation per source, flagging sources that could not fill their quota.")

    for lang, bloco in aloc.items():
        for nota in bloco.get("notas", []):
            print(f"  [{lang}] {nota}")
    if not any(b.get("notas") for b in aloc.values()):
        print("  no deficit redistribution was needed.")
else:
    print("skipped — labeling/manifest.json absent (run `pf make-seed`)")

### The gold slots, and why they carry no mark

Each batch hides 3 gold items among its 80. They are drawn from the hand-revised
calibration batch and **re-sampled across batches**, so the same gold uid reappears in
several batches — which is what makes agreement comparable between annotators who never
shared a batch.

The reuse is deliberately uneven, and the histogram below shows the actual shape. What it
buys: a gold item seen once measures one batch; a gold item seen thirteen times becomes a
reference point that any two annotators can be compared through.

The file handed to an annotator contains only `uid`, `lang` and `text` for all 80 items —
identical fields for gold and non-gold. Which three are gold lives in
`manifest.gold_uids`, on the server side.

In [ ]:
if TEM_MANIFEST and manifesto.get("batches"):
    lotes = manifesto["batches"]
    reuso = Counter()
    for lote in lotes.values():
        for uid in lote.get("gold_uids", []):
            reuso[uid] += 1
    vagas = sum(reuso.values())
    distintos = len(reuso)
    print(f"{len(lotes)} batches x {manifesto.get('gold_per_batch')} gold = {vagas} gold slots, "
          f"filled by {distintos} distinct uids")
    if distintos:
        print(f"mean reuse {vagas / distintos:.2f}x   min {min(reuso.values())}x   "
              f"max {max(reuso.values())}x")

    hist = Counter(reuso.values())
    xs = sorted(hist)
    ys = [hist[x] for x in xs]
    fig, ax = plt.subplots(figsize=(9, 3.4))
    barras = ax.bar([str(x) for x in xs], ys, color=VERDE, width=0.72)
    for b, v in zip(barras, ys, strict=True):
        ax.annotate(str(v), (b.get_x() + b.get_width() / 2, v), ha="center",
                    va="bottom", fontsize=8.5, color="#333")
    ax.set_xlabel("how many batches this gold uid appears in")
    ax.set_ylabel("gold uids")
    ax.set_title(f"Reuse of the gold slots (n = {distintos} distinct uids, "
                 f"{vagas} slots)")
    ax.set_ylim(0, max(ys) * 1.18)
    plt.tight_layout()
    mostrar(fig, "Bar chart histogram of how many batches each distinct gold uid appears in.")

    tam = Counter(lote.get("n_items") for lote in lotes.values())
    novos = Counter(lote.get("n_novos") for lote in lotes.values())
    print("batch sizes  : " + ", ".join(f"{n} items x{c}" for n, c in sorted(tam.items())))
    print("new per batch: " + ", ".join(f"{n} new x{c}" for n, c in sorted(novos.items())))
    total_novos = sum(n * c for n, c in novos.items())
    print(f"new items across all batches: {total_novos:,} "
          f"(+ {manifesto['calibration']['n_items']} in the calibration batch "
          f"= {total_novos + manifesto['calibration']['n_items']:,} of "
          f"{manifesto['n_seed']:,} seed items)")
else:
    print("skipped — no batches in the manifest")

## 2. Campaign status

`agreement` is `None` on a batch that has not been scored, and `None` is **not** `0.0`.
Before any gold has been imported there is no measurement at all, and rendering the absence
as a zero would fail the whole campaign on a number nobody computed. The same distinction
appears in the reviewer calibration of the first notebook — it is the project's recurring
rule about honest absence.

A batch bounced by the agreement gate keeps the score that failed it, but **leaves the
average**: its work was discarded, so counting it would be scoring an annotator on output
nobody kept.

In [ ]:
if TEM_MANIFEST and manifesto.get("batches"):
    lotes = manifesto["batches"]
    estados = Counter(lote.get("status") for lote in lotes.values())
    total = len(lotes)
    feitos = estados.get("done", 0)
    print(f"{feitos} of {total} batches done ({feitos / total:.0%})")
    for st, n in estados.most_common():
        print(f"    {st!s:<12} {n:>4}  ({n / total:.0%})")

    notas = [lote["agreement"] for lote in lotes.values()
             if lote.get("agreement") is not None]
    sem_nota = total - len(notas)
    print(f"\nbatches with an agreement score: {len(notas)} of {total}")
    print(f"batches with agreement = None (not measured): {sem_nota}")

    cal = manifesto.get("calibration", {})
    print(f"\ncalibration batch `{cal.get('batch_id')}`: {cal.get('n_items')} items, "
          f"status = {cal.get('status')!r}, revised_at = {cal.get('revised_at')!r}")
    print(f"manifest.gold (the imported hand-revised labels): {manifesto.get('gold')!r}")

    ordem = ["pending", "claimed", "done", "failed"]
    rot = [s for s in ordem if s in estados] + [s for s in estados if s not in ordem]
    val = [estados[s] for s in rot]
    fig, ax = plt.subplots(figsize=(8.6, 0.6 * len(rot) + 1.8))
    barras = ax.barh(rot, val, color=[VERDE if s == "done" else
                                      VERMELHO if s == "failed" else
                                      LARANJA if s == "claimed" else CINZA for s in rot])
    anotar(ax, val, range(len(rot)), dx=max(val) * 0.02)
    ax.invert_yaxis()
    ax.set_xlim(0, max(val) * 1.2)
    ax.set_xlabel("batches")
    ax.set_title(f"Batch state machine (n = {total} batches)")
    plt.tight_layout()
    mostrar(fig, "Horizontal bar chart of batches by state in the claim state machine: pending, claimed, done, failed.")

    if notas:
        limiar = 0.80
        aprovados = sum(1 for a in notas if a >= limiar)
        fig, ax = plt.subplots(figsize=(9.5, 3.4))
        ax.plot(range(len(notas)), sorted(notas), marker="o", markersize=4,
                linewidth=1.2, color=AZUL)
        ax.axhline(limiar, color=VERMELHO, linestyle="--", linewidth=1.3,
                   label=f"gate = {limiar:.2f} (5 of 6 comparisons)")
        ax.set_xlabel("batches, sorted by agreement")
        ax.set_ylabel("agreement (task_type + domain of 3 golds)")
        ax.set_ylim(0, 1.02)
        ax.set_title(f"Agreement against the gate (n = {len(notas)} scored batches)")
        ax.legend(frameon=False, fontsize=9)
        plt.tight_layout()
        mostrar(fig, "Line chart of per-batch agreement sorted ascending, with a dashed line marking the 0.80 acceptance gate.")
        print(f"{aprovados} of {len(notas)} scored batches clear the {limiar:.2f} gate "
              f"({aprovados / len(notas):.0%})")
        print(f"mean agreement: {sum(notas) / len(notas):.3f}  "
              f"(granularity is 1/6 = {1/6:.3f}; only 7 values are possible)")
    else:
        fig, ax = plt.subplots(figsize=(9.5, 2.4))
        ax.axis("off")
        ax.text(0.5, 0.72, "Agreement: NOT MEASURED YET", ha="center",
                fontsize=13, weight="bold", color=CINZA)
        ax.text(0.5, 0.44, f"0 of {total} batches carry a score. "
                           f"The calibration batch is still `{cal.get('status')}`.",
                ha="center", fontsize=10.5)
        ax.text(0.5, 0.16, "None is not 0.0 — the campaign has not been scored, "
                           "not scored badly.",
                ha="center", fontsize=9.5, color=CINZA, style="italic")
        ax.add_patch(plt.Rectangle((0.02, 0.06), 0.96, 0.88, fill=False,
                                   edgecolor=CINZA, linewidth=1.1, linestyle="--",
                                   transform=ax.transAxes))
        plt.tight_layout()
        mostrar(fig, "Framed text panel stating that campaign agreement has not been measured yet, because no batch carries a score.")
else:
    print("skipped — no batches in the manifest")

### The calibration batch comes from *inside* the 12,000

`batch_0000` is not an extra sample beside the seed — it is the first 100 items **of** it. A
human revises those by hand, `pf labels gold` imports the revision, and only then do the
remaining 11,900 become the working batches. Until that import happens, `manifest.gold` is
`null` and there is nothing to score agreement against; the batch's own status says exactly
that (`gold_pending`).

Once imported, an agent's label on a calibration item is **discarded** by stage s08: each
gold reappears in roughly five batches and exists to measure, not to label.

In [ ]:
if TEM_MANIFEST:
    cal = manifesto.get("calibration", {})
    uids = cal.get("uids") or []
    print(f"calibration batch : {cal.get('batch_id')}")
    print(f"items             : {cal.get('n_items')} (uids present in the manifest: {len(uids)})")
    print(f"status            : {cal.get('status')}")
    print(f"revised_at        : {cal.get('revised_at')}")
    print(f"manifest.gold     : {manifesto.get('gold')}")

    lotes = manifesto.get("batches", {})
    usados = {u for lote in lotes.values() for u in lote.get("gold_uids", [])}
    dentro = len(usados & set(uids))
    print(f"\ngold uids drawn from the calibration batch: {dentro} of {len(usados)} "
          f"({dentro / len(usados):.0%})" if usados else "")
    if cal.get("status") == "gold_pending":
        print("\n  -> `gold_pending` means: the 100 items were sampled and set aside, and the")
        print("     hand revision has NOT been imported yet. Until `pf labels gold --file ...`")
        print("     runs, agreement cannot be computed for any batch, and stage s08 has no")
        print("     manual labels to give precedence to.")
else:
    print("skipped — no manifest")

## 3. What is actually in the seed

Metadata columns only — `uid`, `lang`, `source`, `n_chars`, `native_category`, `bucket`.
The `text` column is never read into this notebook.

Two things worth reading off this section:

* **Stratification by size band is not decoration.** Short prompts and 20,000-character
  prompts are different labeling problems, and a sample drawn without the band would be
  dominated by whichever the biggest source happens to produce.
* **`native_category` coverage is partial, and where it exists it is not a label.** Only
  some sources ship a category of their own, and the mapping to this project's taxonomy is
  deliberately incomplete. The clearest case: `no_robots` uses a single `Generation`
  category for "produce some text", which this taxonomy splits into `geracao-criativa` and
  `redacao-pratica`. That mapping is **`null` on purpose** — guessing one side would inject
  thousands of wrong labels into exactly the two classes that are hardest to tell apart.

In [ ]:
if SEED_PARQUET.exists():
    # NUNCA a coluna `text`: ela carrega o prompt inteiro e é o motivo de a
    # semente ser gitignorada. Ler só o metadado é a política, não uma economia.
    colunas = ["uid", "lang", "source", "n_chars", "native_category", "bucket"]
    t = pq.read_table(SEED_PARQUET, columns=colunas)
    print(f"seed.parquet: {t.num_rows:,} rows, columns read = {colunas}")
    print(f"(the file also has a `text` column, {pq.ParquetFile(SEED_PARQUET).metadata.num_rows:,} "
          f"prompt bodies, which this notebook does not load)")

    langs = t.column("lang").to_pylist()
    fontes = t.column("source").to_pylist()
    buckets = t.column("bucket").to_pylist()
    nativos = t.column("native_category").to_pylist()

    por_lang = Counter(langs)
    print("\nby language: " + ", ".join(f"{k}={v:,}" for k, v in sorted(por_lang.items())))

    par = Counter(zip(fontes, langs, strict=True))
    ordem = sorted(par, key=lambda p: (p[1], -par[p]))
    fig, ax = plt.subplots(figsize=(9.5, 0.44 * len(ordem) + 2.0))
    rot = [f"{f}  ({lg})" for f, lg in ordem]
    val = [par[p] for p in ordem]
    cores = [AZUL if lg == "pt" else LARANJA for _, lg in ordem]
    ax.barh(rot, val, color=cores)
    anotar(ax, val, range(len(rot)), dx=max(val) * 0.015)
    ax.invert_yaxis()
    ax.set_xlim(0, max(val) * 1.18)
    ax.set_xlabel("prompts in the seed")
    ax.set_title(f"Seed composition by source and language (n = {t.num_rows:,})")
    plt.tight_layout()
    mostrar(fig, "Horizontal bar chart of seed composition, one bar per source and language pair, coloured by language.")

    # ---- faixas de tamanho
    pb = Counter(buckets)
    ordem_b = sorted(pb, key=lambda b: min(
        n for n, bb in zip(t.column("n_chars").to_pylist(), buckets, strict=True)
        if bb == b))
    fig, ax = plt.subplots(figsize=(8.4, 2.9))
    val = [pb[b] for b in ordem_b]
    barras = ax.bar(ordem_b, val, color=VERDE, width=0.6)
    for b, v in zip(barras, val, strict=True):
        ax.annotate(f"{v:,}\n{v / t.num_rows:.0%}", (b.get_x() + b.get_width() / 2, v),
                    ha="center", va="bottom", fontsize=9)
    ax.set_ylim(0, max(val) * 1.25)
    ax.set_ylabel("prompts")
    ax.set_xlabel("size band (characters)")
    ax.set_title(f"Seed by size band (n = {t.num_rows:,})")
    plt.tight_layout()
    mostrar(fig, "Bar chart of the seed split into character-size bands, each bar labelled with its count and share.")

    # ---- cobertura de native_category
    com = sum(1 for x in nativos if x)
    print(f"\nnative_category present on {com:,} of {t.num_rows:,} rows "
          f"({com / t.num_rows:.1%}) — the other {t.num_rows - com:,} come from sources "
          f"that publish no category at all")
    top = Counter(x for x in nativos if x).most_common(12)
    if top:
        fig, ax = plt.subplots(figsize=(9, 0.42 * len(top) + 1.8))
        rot = [k for k, _ in top]
        val = [v for _, v in top]
        ax.barh(rot, val, color=CINZA)
        anotar(ax, val, range(len(rot)), dx=max(val) * 0.02)
        ax.invert_yaxis()
        ax.set_xlim(0, max(val) * 1.2)
        ax.set_xlabel("prompts in the seed")
        ax.set_title(f"Most common native_category values (n = {com:,} rows that have one)")
        plt.tight_layout()
        mostrar(fig, "Horizontal bar chart of the most common native_category values across the seed rows that carry one.")

    # ---- os mapeamentos, e o buraco proposital
    mapeamentos = sorted(paths.MAPPINGS.glob("*.json")) if paths.MAPPINGS.exists() else []
    print(f"\nnative category -> task_type mappings on disk: "
          f"{', '.join(p.stem for p in mapeamentos) or '(none)'}")
    for p in mapeamentos:
        mp = json.loads(p.read_text(encoding="utf-8"))
        regras = mp.get("map") or mp.get("mapping") or {}
        nulos = [k for k, v in regras.items() if v is None]
        print(f"    {p.stem}: {len(regras)} native categories, "
              f"{len(nulos)} mapped to null ({', '.join(nulos) or '-'})")
        if mp.get("_unmapped_note"):
            print(f"        note: {mp['_unmapped_note'][:220]}")
else:
    print("skipped — labeling/seed/seed.parquet absent (run `pf make-seed`)")

### The stratification report, as the stage wrote it

`strata.txt` is written by s07 next to the seed. It is the human-readable form of the same
allocation, including the per-source size-band breakdown in `taken/available` form — which
is where an over-drawn band would show up.

It is reproduced **verbatim, in Portuguese**, and so are the `_note` fields of the mapping
files above. That is the project's language convention rather than an oversight: code
comments, docstrings, CLI messages and internal stage artifacts are pt-BR, while everything
*delivered* — dataset card, quality report, audit, and these notebooks — is English.
Translating an internal artifact on the way into a report would mean the report quotes
something that does not exist on disk under that wording.

In [ ]:
if STRATA_TXT.exists():
    print(STRATA_TXT.read_text(encoding="utf-8"))
else:
    print("skipped — labeling/seed/strata.txt absent")

## 4. The labels the campaign will produce

Stage s08 merges the agent labels, the hand-revised gold and the sources' native categories
into `data/final/seed_labels.parquet`, with precedence **manual > agent > native**. Once
that file exists, the class distribution over the taxonomy is the first thing to look at:
the 16 task types and 16 domains are not expected to be uniform, but a class that is empty
in the seed is a class the downstream classifier cannot learn.

In [ ]:
if SEED_LABELS.exists():
    t = pq.read_table(SEED_LABELS)
    print(f"seed_labels.parquet: {t.num_rows:,} rows, columns: {t.schema.names}")
    for eixo in ("task_type", "domain"):
        if eixo not in t.schema.names:
            continue
        vals = t.column(eixo).to_pylist()
        cont = Counter(v for v in vals if v)
        nulos = sum(1 for v in vals if not v)
        classes = list(taxonomia[eixo])           # ordem da taxonomia, zeros incluídos
        val = [cont.get(c, 0) for c in classes]
        fig, ax = plt.subplots(figsize=(9.5, 0.4 * len(classes) + 2.0))
        ax.barh(classes, val, color=AZUL if eixo == "task_type" else VERDE)
        anotar(ax, val, range(len(classes)), dx=(max(val) or 1) * 0.02)
        ax.invert_yaxis()
        ax.set_xlim(0, (max(val) or 1) * 1.2)
        ax.set_xlabel("prompts")
        ax.set_title(f"{eixo} distribution in the seed "
                     f"(n = {t.num_rows - nulos:,} labelled, {nulos:,} unlabelled)")
        plt.tight_layout()
        mostrar(fig, "Horizontal bar chart of the label distribution across the taxonomy axis, drawn over every class in taxonomy order so that empty classes stay visible.")
        vazias = [c for c in classes if not cont.get(c)]
        print(f"{eixo}: {len(classes) - len(vazias)} of {len(classes)} classes populated; "
              f"empty = {', '.join(vazias) or 'none'}")
    if "label_method" in t.schema.names:
        print("\nby precedence: " +
              ", ".join(f"{k}={v:,}" for k, v in
                        Counter(t.column("label_method").to_pylist()).most_common()))
else:
    print("data/final/seed_labels.parquet has NOT been produced yet.")
    print()
    print("It is the output of stage s08, which needs the campaign to have run:")
    print("    1. pf labels gold --file <hand-revised calibration>.jsonl")
    print("    2. run the campaign (skill `rotular-prompts`, or pf labels next / submit)")
    print("    3. pf merge-labels")
    print()
    print("Until then, `task_type`, `domain`, `quality` and `nsfw` are NULL for 100% of the")
    print("corpus, and the curation interface disables those facets with an explanation")
    print("rather than hiding them — a filter group that vanishes reads as a bug.")
    classes = list(taxonomia["task_type"])
    fig, ax = plt.subplots(figsize=(9.5, 0.4 * len(classes) + 2.0))
    ax.barh(classes, [0] * len(classes), color=CINZA)
    ax.set_xlim(0, 1)
    ax.invert_yaxis()
    ax.set_xlabel("prompts")
    ax.set_title(f"task_type distribution — awaiting the campaign "
                 f"(n = 0 of {len(classes)} classes populated)")
    ax.annotate("every class at zero: not measured, not measured as empty",
                xy=(0.5, 0.5), xycoords="axes fraction", ha="center",
                fontsize=10, color=CINZA, style="italic")
    plt.tight_layout()
    mostrar(fig, "Horizontal bar chart of the task_type taxonomy with every class at zero, marking that the labels have not been produced rather than measured as empty.")

## 5. Near-duplicate removal, validated against its own output

This stage **has** run, and its output is on disk. The map
`data/final/dedup_near_map.parquet` records one row per discarded prompt: which canonical
it was folded into, and the cosine and Jaccard measured **against that canonical**.

### Why a pairwise recheck exists at all

Grouping is done with union-find, which is transitive: confirm A~B and B~C and A lands in
the same component as C **without the two ever having been compared**. That is not a
theoretical worry — it was measured on the first run of this stage and it was discarding
good rows. The fix (`[dedup] near_pairwise = true`) rechecks every member **against the
canonical** on all three criteria, and whoever fails stays in the universe.

The historical measurement, from the first execution of s06 and reported in the project's
engineering notes: of 44,333 discards, **27% (11,964) had cosine < 0.985 against their own
group's canonical** and 13.4% had Jaccard < 0.65. Those figures describe a run whose
artifacts are not the ones on disk now and are cited here as history, with attribution, not
recomputed below.

What *is* recomputed below is the invariant that the pairwise recheck guarantees, over the
map that is actually on disk: **every single discarded row clears both thresholds against
its canonical.**

### And why cosine alone must never decide

The embedding model truncates at 512 tokens (~1,900 characters in pt, ~2,100 in en), so two
texts sharing a long header produce identical vectors while diverging completely after the
cut. A real cluster of 2,088 rows shared 8,180 characters of prefix — one asked for a
forecast, another for a travel itinerary. Jaccard is the criterion that sees the whole text.

In [ ]:
import tomllib

cfg = tomllib.loads(paths.SETTINGS_TOML.read_text(encoding="utf-8"))
lim = cfg.get("dedup", {})
COS_MIN, JAC_MIN = lim.get("near_cosine"), lim.get("near_jaccard")
print(f"thresholds from config/settings.toml [dedup]: near_cosine = {COS_MIN}, "
      f"near_jaccard = {JAC_MIN}, near_len_ratio = {lim.get('near_len_ratio')}, "
      f"near_pairwise = {lim.get('near_pairwise')}")
print("(thresholds live in the config file, never hardcoded — it is what makes the stages "
      "replayable)")

# ---- funil por METADATA de parquet: só o footer é lido, nenhuma linha carregada.
print("\nPipeline funnel, read from parquet metadata (no row is loaded):")
etapas = [("raw -> s01 normalize", NORMALIZED), ("s02 language + variant", LANG_PARQUET),
          ("s04 exact dedup", DEDUP1), ("s06 near dedup -> universe", UNIVERSE)]
funil = []
for rotulo, caminho in etapas:
    if caminho.exists():
        n = pq.ParquetFile(caminho).metadata.num_rows
        funil.append((rotulo, caminho.name, n))
        print(f"    {rotulo:<30} {caminho.name:<22} {n:>10,}")
    else:
        print(f"    {rotulo:<30} {caminho.name:<22} {'ABSENT':>10}")

if NEAR_MAP.exists() and DEDUP1.exists() and UNIVERSE.exists():
    n_in = pq.ParquetFile(DEDUP1).metadata.num_rows
    n_out = pq.ParquetFile(UNIVERSE).metadata.num_rows
    n_near = pq.ParquetFile(NEAR_MAP).metadata.num_rows
    resto = n_in - n_near - n_out
    print(f"\n    {n_in:,} into s06")
    print(f"  - {n_near:,} near-duplicates (dedup_near_map.parquet)")
    print(f"  - {resto:,} dropped by the s06 language recheck")
    print(f"  = {n_out:,} in final/universe.parquet")
    assert n_in - n_near - resto == n_out
    print("\nThe language recheck runs BEFORE grouping and uses a window SMALLER than s02's")
    print(f"([dedup] lang_recheck_head_chars = {lim.get('lang_recheck_head_chars')}): a long")
    print("pasted payload drowns the instruction, and the language of a prompt is the")
    print("language of the INSTRUCTION, not of the material pasted underneath it.")

In [ ]:
if NEAR_MAP.exists():
    t = pq.read_table(NEAR_MAP)
    cos = t.column("cosine").to_numpy()
    jac = t.column("jaccard").to_numpy()
    n = len(cos)
    ok_cos = int((cos >= COS_MIN).sum())
    ok_jac = int((jac >= JAC_MIN).sum())
    ok_ambos = int(((cos >= COS_MIN) & (jac >= JAC_MIN)).sum())

    print(f"discarded near-duplicates on disk: {n:,}")
    print(f"  cosine  >= {COS_MIN}: {ok_cos:,} of {n:,}  ({ok_cos / n:.4%})")
    print(f"  jaccard >= {JAC_MIN} : {ok_jac:,} of {n:,}  ({ok_jac / n:.4%})")
    print(f"  BOTH               : {ok_ambos:,} of {n:,}  ({ok_ambos / n:.4%})")
    print("\nINVARIANT: every discarded row clears both thresholds against its canonical — "
          + ("HOLDS." if ok_ambos == n else f"VIOLATED by {n - ok_ambos:,} row(s)."))

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6))
    for ax, dados, minimo, nome, cor in (
        (axes[0], cos, COS_MIN, "cosine", AZUL),
        (axes[1], jac, JAC_MIN, "Jaccard", VERDE),
    ):
        ax.hist(dados, bins=60, color=cor, alpha=0.85)
        ax.axvline(minimo, color=VERMELHO, linestyle="--", linewidth=1.4,
                   label=f"threshold = {minimo}")
        ax.set_xlabel(f"{nome} against the canonical")
        ax.set_ylabel("discarded rows")
        ax.set_title(f"{nome} of the discards (n = {n:,})")
        ax.legend(frameon=False, fontsize=8.5)
        ax.annotate(f"min = {dados.min():.4f}\nmedian = {sorted(dados)[len(dados) // 2]:.4f}\n"
                    f"max = {dados.max():.4f}",
                    xy=(0.03, 0.95), xycoords="axes fraction", va="top",
                    fontsize=8.5, color="#444", family="monospace")
    plt.tight_layout()
    mostrar(fig, "Two histograms side by side, cosine and Jaccard similarity of every discarded near-duplicate against its canonical, each with a dashed line at the configured threshold and the distribution hard-clipped at it.")

    print("Both distributions are hard-clipped at the threshold, which is the visual form of")
    print("the invariant: nothing below the line was ever discarded.")

    canonicos = len(set(t.column("canonical_uid").to_pylist()))
    por_lang = Counter(t.column("lang").to_pylist())
    print(f"\n{n:,} discards folded into {canonicos:,} distinct canonicals "
          f"({n / canonicos:.1f} discards per canonical on average)")
    print("by language: " + ", ".join(f"{k}={v:,}" for k, v in por_lang.most_common()))
    print("\nThe canonical never changes when members are spared: `dedup.choose_canonical` is")
    print("a min over the group and validation only ever removes non-canonical rows — the")
    print("minimum of a subset that still contains the minimum is the same minimum. That is")
    print("what makes validating in a single pass correct, and it is an invariant, not luck.")
else:
    print("skipped — data/final/dedup_near_map.parquet absent (run `pf run s06`)")

## 6. Exact duplicates, and the tiebreak nobody sees

Stage s04 folds byte-identical normalized texts by `hash_norm`. Which row survives is
**not** arbitrary: the tiebreak is the order of the sections in `config/sources.toml`. That
is the reason a config file's *ordering* is documented as load-bearing — the same corpus
with the sections shuffled would keep a different row from each duplicate family, and the
licence carried by the surviving row would change with it.

The cross-source column below is where that matters: when a duplicate spans two sources,
the tiebreak decides which licence the corpus keeps.

In [ ]:
if EXACT_MAP.exists():
    t = pq.read_table(EXACT_MAP)
    n = t.num_rows
    src = t.column("source").to_pylist()
    can = t.column("canonical_source").to_pylist()
    cruzados = [(a, b) for a, b in zip(src, can, strict=True) if a != b]
    print(f"exact duplicates removed: {n:,}")
    print(f"  same source as the canonical : {n - len(cruzados):,} "
          f"({(n - len(cruzados)) / n:.1%})")
    print(f"  cross-source                 : {len(cruzados):,} ({len(cruzados) / n:.1%})"
          "   <- here the sources.toml order decides which licence survives")

    perdedores = Counter(a for a, _ in cruzados)
    vencedores = Counter(b for _, b in cruzados)
    if cruzados:
        fontes = sorted(set(perdedores) | set(vencedores))
        y = range(len(fontes))
        alt = 0.38
        fig, ax = plt.subplots(figsize=(9.5, 0.5 * len(fontes) + 2.0))
        v1 = [perdedores.get(f, 0) for f in fontes]
        v2 = [vencedores.get(f, 0) for f in fontes]
        ax.barh([i + alt / 2 for i in y], v1, alt, color=VERMELHO, label="row discarded from")
        ax.barh([i - alt / 2 for i in y], v2, alt, color=VERDE, label="canonical kept from")
        for i, (a, b) in enumerate(zip(v1, v2, strict=True)):
            if a:
                ax.annotate(f"{a:,}", (a + max(v1 + v2) * 0.01, i + alt / 2),
                            va="center", fontsize=8, color="#333")
            if b:
                ax.annotate(f"{b:,}", (b + max(v1 + v2) * 0.01, i - alt / 2),
                            va="center", fontsize=8, color="#333")
        ax.set_yticks(list(y), fontes)
        ax.invert_yaxis()
        ax.set_xlim(0, max(v1 + v2) * 1.2)
        ax.set_xlabel("cross-source exact duplicates")
        ax.set_title(f"Which source loses, which keeps the canonical "
                     f"(n = {len(cruzados):,} cross-source pairs)")
        ax.legend(frameon=False, fontsize=8.5, loc="lower right")
        plt.tight_layout()
        mostrar(fig, "Horizontal bar chart pairing, per source, how many rows it lost to a cross-source exact duplicate against how many canonicals it kept.")
    else:
        print("\n  no cross-source exact duplicate in this run — every family stayed inside "
              "one source, so the tiebreak never had to choose a licence.")

    familias = len(set(t.column("canonical_uid").to_pylist()))
    print(f"\n{n:,} discards folded into {familias:,} distinct canonicals")
    print("Note the empty-key case that does NOT group: norm_for_hash('!!!') == '', so every")
    print("punctuation-only prompt shares the sha256 of the empty string. s04 lets those rows")
    print("through whole rather than collapsing them into one.")
else:
    print("skipped — data/interim/dedup_exact_map.parquet absent (run `pf run s04`)")

## 7. What this cannot claim yet

* **The labeling campaign has not run.** Every batch is `pending`, `manifest.gold` is
  `null`, and the calibration batch is `gold_pending`. What is validated above is the
  campaign's **design and bookkeeping** — quotas, deficit redistribution, gold reuse, the
  batch state machine — not its output. There is no agreement number, no inter-annotator
  measurement and no class distribution, and the panels say "not measured" rather than
  drawing a zero.
* **Therefore the corpus taxonomy is still empty.** `task_type`, `domain`, `quality` and
  `nsfw` are NULL for 100% of the universe. Any claim about what kinds of prompt this corpus
  contains is, today, unsupported by labels.
* **The near-duplicate invariant is a consistency check, not a quality judgement.** It
  proves that every discarded row met the configured thresholds against its canonical. It
  does **not** prove the thresholds are the right ones — that is a human decision, which is
  why `pf report dedup-sample` writes 50 pairs to a file for a person to read.
* **Transitivity is mitigated, not eliminated.** Two spared members may still be duplicates
  of each other. Re-grouping them would require running union-find to a fixed point with the
  canonical shifting mid-flight; the project deliberately prefers letting a few near pairs
  through to discarding good rows by transitivity. The trade is a choice, and it is recorded
  as one.
* **The historical 44,333 / 27% figures are not reproducible from the artifacts on disk.**
  They describe the first execution of s06, before the pairwise recheck existed. They are
  cited as history, attributed as such, and no chart above is drawn from them.

---

### Licence and provenance of this page

`labeling/seed/seed.parquet` and `labeling/batches/*.json` carry full prompt bodies and are
gitignored for that reason; the sources behind them include CC-BY-NC-4.0 material. This
notebook reads the seed's **metadata columns only** and never loads a batch file, and the
rendered page contains **counts and distributions exclusively** — no prompt text of any
length, from any source. Per-row licence and attribution for actual data exports are emitted
by `prompt_factory.export`, resolved from `config/sources.toml`.